# Use governed community stubs safely

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/09_community_stubs.ipynb)

Community stubs are declarative manifests, not executable Python. TensorGuard validates provenance and executes conformance cases before a third-party layer can turn UNKNOWN into a precise check.

In [ ]:
%pip install -q tensorguard  # on Colab; locally: pip install -e .

In [ ]:
from src.stub_governance import load_community_stubs, validate_directory
from src.shape_stub_registry import clear_user_stubs, get_shape_stub
from src.tensor_shapes import ShapeDim, TensorShape

reports = validate_directory('../../community_stubs')
print('valid manifests:', len([r for r in reports if r.ok]), '/', len(reports))
assert reports and all(r.ok for r in reports)

loaded = load_community_stubs('../../community_stubs')
print('loaded stubs:', loaded)
assert 'Linear8bitLt' in loaded
stub = get_shape_stub('Linear8bitLt')
params = stub.bind_params((768, 3072), {})
out, err = stub.transfer(TensorShape((ShapeDim('batch'), ShapeDim(768))), params)
assert err is None and out.dims[-1].value == 3072
_, err = stub.transfer(TensorShape((ShapeDim('batch'), ShapeDim(512))), params)
assert err and '768' in err
clear_user_stubs()